# 6. Teaching a Computer to Guess
### Stage 6 of 7 — building and honestly testing a prediction model

---

**How to read this notebook:** This is part of a 7-notebook series that walks through the *entire* research project, step by step, in the same order the actual analysis was done — this one covers stage 6 of 7. Every number and chart here is real, pulled directly from the study's actual data (130 students, 13 puzzles). A few small grey boxes contain code — you don't need to understand the code itself, just run each one (click it, then press Shift+Enter) to see the real result. Nothing needs to be edited.

---

## Overview

Notebook 5 showed real, statistically solid differences between High- and Low-performing students. This stage asks the practical follow-up question: **if you gave a computer program only a student's eye-movement measurements — never their actual results — could it correctly guess whether they were High- or Low-performing?**

This is exactly the kind of task **machine learning** is built for: a program that studies many past examples (in this case, all 104 "training" students' measurements *and* their actual results) and learns which patterns tend to go together, so it can make an educated guess about someone new.

## What You'll Learn Here

1. Why having too many measurements relative to the number of students is itself a problem, and what was done about it
2. Why testing a program fairly requires holding some real data back and never looking at it until the very end
3. How five different learning methods were compared on equal footing
4. How to check a "good" result isn't secretly a lucky guess
5. How a prediction gets turned into a final Yes/No decision

## Background, Explained Simply

### Too many measurements, not enough students

There were **416 measurements** but only **104 students** available for building the model. That's an awkward ratio — with that many measurements to choose from, a learning program can accidentally "discover" a pattern that's really just coincidence in this particular group of 104 people, one that wouldn't hold up on anyone else. Two things were done about this:

- **Removing exact duplicates and near-duplicates.** Many measurements turned out to be mathematically near-identical to each other (for example, "average time on wrong spots" and "total time on wrong spots" move in lockstep whenever there's only one wrong-spot region to measure). Grouping these near-duplicate measurements together and keeping just one representative from each group brought the count down from 416 to **240** — same information, much less redundancy.
- **Letting the computer pick a further, smaller shortlist automatically**, separately for every test, so the exact shortlist can adapt rather than being fixed in advance.

### Keeping the "final exam" group genuinely untouched

This is worth explaining carefully, because getting it wrong is a classic, easy-to-make mistake. **26 students were set aside back in Notebook 4 and never used for anything — not for choosing measurements, not for picking the learning method, not for fine-tuning it — until the single, final check at the very end of this notebook.**

An earlier version of this exact study actually got this wrong: the "final exam" students had accidentally been included in the fine-tuning process. It's an easy mistake to make (all 130 students' data lives in one file, so it's simple to forget which 26 rows are supposed to be off-limits), and it makes a model look better than it really is — like grading yourself using the answer key you used to study. **This was caught and fixed**, and the fix is one of the things this notebook demonstrates.

### Comparing five learning methods fairly

Five different methods were tried, evaluated the same way for fairness: split the 104 training students into 5 groups, train on 4 of them, test on the 1 left out — and repeat 5 times so every student gets a turn as the "unseen" test case. Averaging across all 5 rounds gives an honest per-method score:

| Method | How it makes decisions |
|---|---|
| **Always guess the most common answer** (the baseline) | Doesn't look at the data at all — a sanity-check floor that any real method should beat |
| **Logistic Regression** | Adds up weighted evidence for and against, like a simple scorecard |
| **Decision Tree** | Asks a small number of yes/no questions in sequence, like a flowchart |
| **Random Forest** | Builds many different flowcharts and lets them vote |
| **Support Vector Machine** | Finds the clearest possible dividing line between the two groups |

### Was the winning result just a lucky guess?

Any pattern-finding program can look impressive by accident. The stress test: **shuffle which student actually got which real result, completely at random, 1,000 times**, and see how often a method can "guess" that well purely from a random shuffle. If it rarely or never manages it, the real, unshuffled result is very unlikely to be a fluke.

### Turning a probability into a Yes/No answer

A learning method doesn't just output "High" or "Low" — internally, it outputs something more like *"I'm 62% confident this student is High-performing."* Somewhere, a cutoff has to be chosen: above this percentage, call it "High"; below it, call it "Low." The default cutoff is 50%, but that's not automatically the *best* cutoff — this notebook checks whether a different cutoff does better, particularly for making sure struggling and strong students are caught at similar rates.

## Discussion Questions

1. Why is having 416 measurements for only 104 students a genuine risk, rather than just "more information is always better"? What could go wrong specifically?
2. A model scores 90% on the students it was trained on, but only 55% on brand-new students. What does that gap suggest, and how would the "final exam" group described above help you catch this?
3. The stress test shuffled real results randomly 1,000 times. Why is that a more convincing check than simply asking "is 81% a good score?"
4. If the model catches struggling students very reliably but misses many strong students, would you rather move the Yes/No cutoff, or would that trade-off just move the problem somewhere else? What would you want to know before deciding?

---

## Seeing the Real Comparison

In [ ]:
# Run this cell to compare all 5 learning methods, before any fine-tuning
from IPython.display import Image
Image(filename='../outputs/figures/06_model_comparison.png')

## The Headline Result

After removing redundant measurements (416 → 240) and carefully fine-tuning all four real methods (not just whichever looked best at first glance — an earlier check found that the method that looks best *before* tuning isn't reliably the one that tunes *best*), the **Decision Tree** came out ahead:

| Method | Balanced accuracy after fine-tuning |
|---|---|
| Logistic Regression | 73.1% |
| **Decision Tree** | **80.9%** |
| Random Forest | 75.4% |
| Support Vector Machine | 72.5% |
| *(A "committee vote" of the top 3 methods)* | *(79.3% — good, but not quite as good as the Decision Tree alone)* |

> **The winning method correctly identified whether a student was High- or Low-performing about 4 times out of 5 (80.9% balanced accuracy), averaged across the 5 rounds of testing on the 104 training students.**

Interestingly, a "committee vote" combining the top 3 methods was tried too — and while it did reasonably well (79.3%), it didn't beat the single Decision Tree, so the simpler method won out.

## Was That Just a Lucky Guess?

In [ ]:
# Run this cell to see the result of randomly shuffling the answers 1,000 times
from IPython.display import Image
Image(filename='../outputs/figures/06_permutation_test.png')

Out of 1,000 completely random shuffles, essentially none scored as well as the real, unshuffled pattern — only about **1 in 1,000** (p = 0.001). That's strong evidence this is a genuine, learnable pattern.

## Choosing the Decision Cutoff

In [ ]:
# Run this cell to see how the Yes/No cutoff was chosen
from IPython.display import Image
Image(filename='../outputs/figures/06_threshold_tuning.png')

In this case, adjusting the cutoff away from the default 50% didn't actually improve the balanced result any further (both landed at essentially the same 81.1% score, evaluated fairly on data the model hadn't been fit to) — so the default cutoff turned out to already be reasonable here. That's a useful, honest finding in itself: not every adjustment you try will turn out to help, and this notebook checked rather than assumed.

## The Real Final Exam: 26 Untouched Students

Every number above came from the 104 training students. Now, for the first and only time, the finished, tuned Decision Tree is tested on the **26 students it has never seen in any form**:

| | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| Low | 0.67 | 0.91 | 0.77 | 11 |
| High | 0.91 | 0.67 | 0.77 | 15 |
| **Overall accuracy** | | | **77%** (20 of 26 correct) | 26 |

## Seeing Every Guess, One by One: the Confusion Matrix

The precision/recall table above is a summary — the **confusion matrix** underneath it shows the actual raw count of every single guess the model made on these 26 students, split out by what really happened vs. what the model predicted:

In [ ]:
# Run this cell to see the confusion matrix (and ROC curve) on the 26 untouched students
from IPython.display import Image
Image(filename='../outputs/figures/06_confusion_roc.png')

**How to read this chart:** the left panel is the confusion matrix — each of the 26 real students falls into exactly one of four boxes, based on what they actually were vs. what the model guessed:

| | Model guessed: Low | Model guessed: High |
|---|---|---|
| **Really was: Low** (11 students) | **10** ✅ correctly caught | **1** ❌ wrongly called High |
| **Really was: High** (15 students) | **5** ❌ wrongly called Low | **10** ✅ correctly caught |

Reading it this way makes the earlier recall numbers concrete: of the 11 students who really were Low-performing, the model correctly caught **10 of them** (91%) and only missed 1. Of the 15 students who really were High-performing, it correctly caught **10 of them** (67%) but mistakenly called the other **5** "Low." That gap — 1 mistake on one side, 5 mistakes on the other — is the recall asymmetry in its most concrete, countable form: not an abstract percentage, but 5 real students whose actual strong performance the model didn't recognise.

The right panel (the **ROC curve**) shows the same trade-off a different way — how well the model separates the two groups across *every possible* Yes/No cutoff, not just the one it happened to use. A curve that hugs the top-left corner would mean near-perfect separation; the diagonal dashed line marks "no better than a coin flip." This model's curve sits solidly above the diagonal, consistent with the honest 77% hold-out accuracy — clearly better than guessing, but far from perfect.

Reading the whole hold-out section honestly: on these 26 genuinely new students, the model **correctly identified 10 of 11 Low-performers** (91% recall) but only **10 of 15 High-performers** (67% recall) — it's noticeably more reliable at catching struggling students than at catching strong ones. That's a real, useful thing to know about this model's behaviour, not something to gloss over.

---

**Next: [07_what_did_it_learn.ipynb](07_what_did_it_learn.ipynb)** — now that we have a working (if imperfect) prediction model, what clues is it actually relying on, and why?